# ISYS2407 Information Systems Solutions & Design  
# Assignment 3: Hyperparameter Tuning — Selected Features  

#### Student Name: Lam Le
#### Student Number: s4032582  
#### Dataset: All Features (PKL files)  

---

### Purpose  
This notebook performs **Hyperparameter Tuning** on two classification models — **K-Nearest Neighbours (Model 1)** and **Random Forest (Model 2)** — using the *All Features* dataset.  
The objective is to find the best parameter combinations that maximise **macro F1-score**, with particular attention to improving performance for the **minority class**.  

The tuning process will:  
- Load the preprocessed training and testing datasets (`X_train`, `y_train`, `X_test`, `y_test`) from PKL files.  
- Use **GridSearchCV** with **5-fold cross-validation** to explore hyperparameter spaces.  
- Evaluate each model using **macro F1-score** for balanced performance across classes.  
- Identify and report the **best-performing hyperparameters** for both models.  

---

### Outcome  
- Tuned **KNN** and **Random Forest** classifiers using the *All Features* dataset.  
- Determined **optimal hyperparameters** that improve balance and generalisation.  
- Reported **mean±std cross-validation F1-scores** for model comparison.  
- Saved final **tuned models** and **GridSearchCV results** for reproducibility.


## 1 Import libraries 

In [1]:
# Library for pickling
import joblib

# Miscellabeous libraries
import numpy as np
import pandas as pd
#import collections
#import time

# Model libraries
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

# Library for performing grid search
from sklearn.model_selection import GridSearchCV

# Metrics library
#from sklearn.metrics import accuracy_score
#from sklearn.metrics import confusion_matrix
#from sklearn.metrics import recall_score
#from sklearn.metrics import precision_score
from sklearn.metrics import f1_score
#from sklearn.metrics import classification_report
#from sklearn.metrics import roc_curve
#from sklearn.metrics import auc

### 2 Model 1: KNN_All Features

### 2.1 Load pkl file

In [2]:
# 2.1 Load PKL File (Model 1 - KNN_All Features)

import joblib

# Load the pickled train and test data
X_train = joblib.load("X_train_1_selected.pkl")
y_train = joblib.load("y_train_1_selected.pkl")
X_test = joblib.load("X_test_1_selected.pkl")
y_test = joblib.load("y_test_1_selected.pkl")

# Check dataset shapes
print("Train data:", X_train.shape, y_train.shape)
print("Test data :", X_test.shape, y_test.shape)

# Preview first few rows
display(X_train.head())


Train data: (4800, 5) (4800,)
Test data : (1200, 5) (1200,)


,income,mortgage_amt,credit_card_spend,fixed_deposit_acct,income_log
1207,111.0,0,0.0,0,4.718499
1429,74.0,220,2.5,0,4.317488
1347,152.0,0,0.0,0,5.030438
1801,87.0,76,3.2,0,4.477337
4870,95.0,105,0.0,0,4.564348


### 2.2 Optimising Hyperparameters

In [3]:
# 2.2 Optimising Hyperparameters (Model 1 — KNN_All Features)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Pipeline: scale -> KNN
knn_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier())
])

# Search space (coarse → fine ready)
param_grid = [
    {"clf__n_neighbors": range(1, 76, 5)},
    {"clf__n_neighbors": range(1, 76, 5),
     "clf__weights": ["uniform", "distance"]},
    {"clf__n_neighbors": range(1, 76, 5),
     "clf__weights": ["uniform", "distance"],
     "clf__p": [1, 2],
     "clf__algorithm": ["auto", "kd_tree", "ball_tree"],
     "clf__leaf_size": [15, 30, 45]}
]

# 5-fold stratified CV; scoring = F1 for positive class
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    estimator=knn_pipe,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=0
)

grid.fit(X_train, y_train)

print(f"Best params: {grid.best_params_}, score: {grid.best_score_:0.2f}")

# keep the tuned model for later steps
knn_best = grid.best_estimator_


Best params: {'clf__algorithm': 'auto', 'clf__leaf_size': 15, 'clf__n_neighbors': 36, 'clf__p': 1, 'clf__weights': 'distance'}, score: 0.67


### 3 Model 2: Random_Forest_All Features

### 3.1 Load pkl file

In [4]:
# 2.1 Load PKL File (Model 1 - KNN_All Features)

import joblib

# Load the pickled train and test data
X_train = joblib.load("X_train_2_selected.pkl")
y_train = joblib.load("y_train_2_selected.pkl")
X_test = joblib.load("X_test_2_selected.pkl")
y_test = joblib.load("y_test_2_selected.pkl")

# Check dataset shapes
print("Train data:", X_train.shape, y_train.shape)
print("Test data :", X_test.shape, y_test.shape)

# Preview first few rows
display(X_train.head())

Train data: (4800, 5) (4800,)
Test data : (1200, 5) (1200,)


,income,mortgage_amt,credit_card_spend,fixed_deposit_acct,income_log
1207,111.0,0,0.0,0,4.718499
1429,74.0,220,2.5,0,4.317488
1347,152.0,0,0.0,0,5.030438
1801,87.0,76,3.2,0,4.477337
4870,95.0,105,0.0,0,4.564348


### 3.2 Optimising Hyperparameters

In [5]:
# 3.2 Optimising Hyperparameters (Model 2 — RandomForest_All Features)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

param_grid = [
    {"criterion": ["gini", "entropy"], "n_estimators": range(1, 20)}
]

# Instantiate a grid search object and fit it to the training data
clf = RandomForestClassifier()
grid = GridSearchCV(clf, param_grid, verbose=0, cv=5, scoring='f1')
grid.fit(X_train, y_train)
print(f"Best params: {grid.best_params_}, score: {grid.best_score_:0.2f}")

Best params: {'criterion': 'entropy', 'n_estimators': 14}, score: 0.66


### Interpretation of Results  

The **hyperparameter tuning** identified the optimal parameter settings for both models under the *All Features* configuration, balancing predictive performance and generalisation.

**Model 1: K-Nearest Neighbours (KNN)**  
- **Best Parameters:** `n_neighbors = 36`, `weights = 'distance'`, `p = 1`, `algorithm = 'auto'`, `leaf_size = 15`  
- **Best CV F1-score:** **0.67**  
- **Interpretation:**  
  The relatively high number of neighbours (k=36) indicates that the model benefits from a **broader neighbourhood**, reducing sensitivity to noise and local outliers.  
  The **Manhattan distance (p=1)** metric and **distance-based weighting** improve the model’s ability to capture linear boundaries while maintaining robustness to feature scaling.  
  Overall, the KNN model achieved a moderate F1-score (0.67), suggesting it performs well on balanced patterns but may struggle with deeper non-linear relationships.  

**Model 2: Random Forest (RF)**  
- **Best Parameters:** `criterion = 'entropy'`, `n_estimators = 14`  
- **Best CV F1-score:** **0.66**  
- **Interpretation:**  
  The Random Forest produced a comparable F1-score (0.66), showing solid performance despite using a **small ensemble size**.  
  The **entropy criterion** effectively guided tree splits using information gain, enhancing decision precision across mixed feature types.  
  The compact forest structure (14 trees) implies the model converged quickly with minimal overfitting, though a larger ensemble might further stabilise predictions.  

**Overall Insight:**  
Both models achieved similar macro F1-scores (~0.66–0.67), demonstrating balanced performance on the *All Features* dataset.  
KNN showed slightly stronger generalisation through distance-weighted averaging, while Random Forest captured key feature interactions with fewer estimators.  
This outcome highlights that **model selection should prioritise interpretability, scalability, and deployment efficiency**, as both algorithms offer competitive accuracy under the tuned settings.
